<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_09_decorators/live_coding_decorators_blog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Декоратори в Python: наскрізний приклад
### Від дублювання коду до декоратора — на прикладі розмежування доступу в блозі

---
## Вступ

Розглянемо простий блог. У блозі є дії: переглядати пости, редагувати, видаляти, публікувати.

І є **три ролі** користувачів:
- `guest` — гість, може тільки читати
- `user` — звичайний користувач
- `admin` — адміністратор, може все

Завдання: кожна дія має перевіряти, чи є у користувача право її виконувати.

---
## 1. Наївна реалізація без декораторів

Найпростіший спосіб перевірити права доступу — додати `if`-перевірку на початку кожної функції.

In [8]:
# Поточний користувач — можна змінювати для тестування
current_user = {}

In [9]:
def view_post(post_id):
    # Гості теж можуть читати — дозволяємо всім
    if current_user["role"] not in ["guest", "user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


def create_post(title):
    # Гостям не можна створювати пости
    if current_user["role"] not in ["user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"✍️  {current_user['name']} створює пост: '{title}'")


def edit_post(post_id):
    # Тільки user і admin
    if current_user["role"] not in ["user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


def publish_post(post_id):
    # Публікувати може тільки admin
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


def delete_post(post_id):
    # Видаляти може тільки admin
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


def archive_post(post_id):
    # Архівувати може тільки admin
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"📁  {current_user['name']} архівує пост #{post_id}")

In [10]:
# Тестуємо як гість
current_user = {"name": "Іван", "role": "guest"}

view_post(1)
create_post("Мій перший пост")
delete_post(1)

👁️  Іван переглядає пост #1
❌ Доступ заборонено!
❌ Доступ заборонено!


In [11]:
# Тестуємо як admin
current_user = {"name": "Оля", "role": "admin"}

view_post(1)
create_post("Важливе оголошення")
delete_post(1)

👁️  Оля переглядає пост #1
✍️  Оля створює пост: 'Важливе оголошення'
🗑️  Оля видаляє пост #1


---
## 2. Проблема: дублювання логіки перевірки

Код працює, але перевірка ролі повторюється в кожній функції:

```python
if current_user["role"] not in [...]:
    print("❌ Доступ заборонено!")
    return
```

Цей шматок зустрічається **6 разів** — у 6 функціях.

Якщо функцій не 6, а 60, правку довелось б вносити в 60 місцях. Те саме стосується зміни тексту повідомлення про помилку або додавання логування невдалих спроб доступу: кожна зміна вимагає редагування коду в кожній функції окремо.

Це дублювання коду — логіка доступу **розкидана** по всіх функціях замість того, щоб бути в одному місці.

---
## 3. Ускладнення вимог: нова роль

Вимога змінюється: потрібна нова роль `moderator`, яка може редагувати і публікувати, але не може видаляти.

Оскільки перевірка доступу продубльована в кожній функції, цю зміну доведеться вносити вручну в кожній з них:

In [6]:
# Нам ВРУЧНУ треба зайти в кожну функцію і додати moderator

def view_post_v2(post_id):
    if current_user["role"] not in ["guest", "user", "admin", "moderator"]:  # ← додали
        print("❌ Доступ заборонено!")
        return
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


def edit_post_v2(post_id):
    if current_user["role"] not in ["user", "admin", "moderator"]:  # ← додали
        print("❌ Доступ заборонено!")
        return
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


def publish_post_v2(post_id):
    if current_user["role"] not in ["admin", "moderator"]:  # ← додали
        print("❌ Доступ заборонено!")
        return
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


def delete_post_v2(post_id):
    if current_user["role"] not in ["admin"]:  # ← moderator НЕ може видаляти!
        print("❌ Доступ заборонено!")
        return
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")

# ... і так далі ще в 2 функціях

print("Ми тільки що вручну зайшли в 4+ функції заради ONE нової ролі 🤦")

Ми тільки що вручну зайшли в 4+ функції заради ONE нової ролі 🤦


Ось картина після зміни:

| Функція | Хто має доступ |
|---------|----------------|
| `view_post` | guest, user, admin, **moderator** |
| `create_post` | user, admin |
| `edit_post` | user, admin, **moderator** |
| `publish_post` | admin, **moderator** |
| `delete_post` | admin |
| `archive_post` | admin |

Щоразу, коли бізнес-правила змінюються, доведеться знову вручну редагувати кожну функцію. Такий підхід не масштабується.

---
## 4. Ідея рішення: функція-обгортка

Потрібен спосіб «загорнути» будь-яку функцію в перевірку доступу:

```
ПЕРЕД виконанням функції → перевір роль
  якщо роль підходить → виконай функцію
  якщо ні → заблокуй
```

Тобто потрібна **обгортка** — функція, яка приймає іншу функцію і додає до неї перевірку доступу. Побудуємо її крок за кроком.

In [7]:
# Крок 1: Функції є об'єктами в Python
# Їх можна передавати як аргументи!

def say_hello():
    print("Привіт!")


def run_function(func):
    print("--- Запускаємо функцію ---")
    func()  # викликаємо функцію, яку передали
    print("--- Готово ---")


run_function(say_hello)  # передаємо say_hello БЕЗ дужок!

--- Запускаємо функцію ---
Привіт!
--- Готово ---


In [10]:
# Крок 2: Функція може ПОВЕРТАТИ іншу функцію
# Це трохи незвично, але дуже потужно

def create_greeter(name):
    # Всередині визначаємо нову функцію
    def greet():
        print(f"Привіт, {name}!")
    # І повертаємо її (без виклику!)
    return greet


greet_ivan = create_greeter("Іван")
greet_olia = create_greeter("Оля")

greet_ivan()
greet_olia()

Привіт, Іван!
Привіт, Оля!


In [11]:
# Крок 3: Поєднуємо обидві ідеї
# Приймаємо функцію → повертаємо нову функцію з перевіркою

def require_admin(func):
    """Обгортка: перевіряє роль перед виконанням"""

    def wrapper():
        # Ось вся логіка перевірки — в одному місці!
        if current_user["role"] != "admin":
            print("❌ Доступ заборонено! Потрібна роль: admin")
            return
        # Якщо все ок — виконуємо оригінальну функцію
        func()

    return wrapper


# Тестуємо
def delete_everything():
    print("🗑️ Видалено все!")


# «Загортаємо» функцію в захист
safe_delete = require_admin(delete_everything)

current_user = {"name": "Гість", "role": "guest"}
safe_delete()  # заблоковано

current_user = {"name": "Адмін", "role": "admin"}
safe_delete()  # дозволено

❌ Доступ заборонено! Потрібна роль: admin
🗑️ Видалено все!


Логіка доступу тепер **в одному місці**.

Саме це і є ідея декоратора. Python дає для неї **зручний синтаксис**.

---
## 5. Синтаксис декоратора: `@`

Те, що зроблено вище, — це і є декоратор. Python дозволяє записати це через `@`.

Замість:
```python
safe_delete = require_admin(delete_everything)
```

Можна писати:
```python
@require_admin
def delete_everything():
    ...
```

Це **рівно одне й те саме** — `@` це синтаксичний цукор.

---
## 6. Універсальний декоратор `require_role`

Декоратор `require_admin` перевіряє тільки одну роль. Зробимо універсальний декоратор — для будь-якого набору ролей.

In [1]:
import functools


def require_role(*allowed_roles):
    """
    Декоратор-фабрика: приймає список дозволених ролей
    і повертає декоратор, який перевіряє роль поточного користувача.
    """
    def decorator(func):
        @functools.wraps(func)  # зберігаємо ім'я та документацію оригінальної функції
        def wrapper(*args, **kwargs):
            # Перевіряємо роль — ВСЯ логіка тут, в одному місці
            if current_user["role"] not in allowed_roles:
                print(f"❌ Доступ заборонено! Потрібна роль: {' або '.join(allowed_roles)}")
                return
            # Роль підходить — викликаємо оригінальну функцію
            # *args, **kwargs — передаємо ВСІ аргументи, які прийшли у wrapper
            return func(*args, **kwargs)

        return wrapper
    return decorator

**Розберемо по частинах:**

| Що | Навіщо |
|----|--------|
| `require_role(*allowed_roles)` | Зовнішня функція — приймає список ролей (`"admin"`, `"user"`, ...) |
| `decorator(func)` | Середня функція — приймає функцію, яку «захищаємо» |
| `wrapper(*args, **kwargs)` | Внутрішня функція — те, що реально виконується замість оригіналу |
| `*args, **kwargs` | «Передай усе далі» — щоб декоратор підходив для будь-якої функції |
| `@functools.wraps(func)` | Щоб `delete_post.__name__` лишався `"delete_post"`, а не `"wrapper"` |

---
## 7. Рефакторинг застосунку

Перепишемо всі функції блогу через декоратор `require_role`.

In [2]:
# ✅ ПІСЛЯ РЕФАКТОРИНГУ — логіка доступу прибрана з тіла функцій

@require_role("guest", "user", "admin", "moderator")
def view_post(post_id):
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


@require_role("user", "admin")
def create_post(title):
    print(f"✍️  {current_user['name']} створює пост: '{title}'")


@require_role("user", "admin", "moderator")
def edit_post(post_id):
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


@require_role("admin", "moderator")
def publish_post(post_id):
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


@require_role("admin")
def delete_post(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


@require_role("admin")
def archive_post(post_id):
    print(f"📁  {current_user['name']} архівує пост #{post_id}")


print("Декоратори задекларовано ✅")

Декоратори задекларовано ✅


In [3]:
# Тестуємо — гість
current_user = {"name": "Гість Анонім", "role": "guest"}
print(f"=== Користувач: {current_user['name']} (роль: {current_user['role']}) ===")

view_post(42)
create_post("Мій пост")
delete_post(42)

=== Користувач: Гість Анонім (роль: guest) ===
👁️  Гість Анонім переглядає пост #42
❌ Доступ заборонено! Потрібна роль: user або admin
❌ Доступ заборонено! Потрібна роль: admin


In [4]:
# Тестуємо — модератор
current_user = {"name": "Марко-Модератор", "role": "moderator"}
print(f"=== Користувач: {current_user['name']} (роль: {current_user['role']}) ===")

view_post(42)
edit_post(42)
publish_post(42)
delete_post(42)   # ← не зможе

=== Користувач: Марко-Модератор (роль: moderator) ===
👁️  Марко-Модератор переглядає пост #42
✏️  Марко-Модератор редагує пост #42
📢  Марко-Модератор публікує пост #42
❌ Доступ заборонено! Потрібна роль: admin


In [5]:
# Тестуємо — адмін
current_user = {"name": "Адмін Всесильний", "role": "admin"}
print(f"=== Користувач: {current_user['name']} (роль: {current_user['role']}) ===")

view_post(42)
create_post("Важливий анонс")
edit_post(42)
publish_post(42)
delete_post(42)
archive_post(42)

=== Користувач: Адмін Всесильний (роль: admin) ===
👁️  Адмін Всесильний переглядає пост #42
✍️  Адмін Всесильний створює пост: 'Важливий анонс'
✏️  Адмін Всесильний редагує пост #42
📢  Адмін Всесильний публікує пост #42
🗑️  Адмін Всесильний видаляє пост #42
📁  Адмін Всесильний архівує пост #42


---
## 8. Порівняння «до» і «після»

**До рефакторингу** — функція `delete_post`:
```python
def delete_post(post_id):
    if current_user["role"] not in ["admin"]:   # ← ця логіка
        print("❌ Доступ заборонено!")            # ← повторюється
        return                                   # ← 6 разів
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")
```

**Після рефакторингу** — функція `delete_post`:
```python
@require_role("admin")             # ← одна строчка, все зрозуміло
def delete_post(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")
```

Тіло функції тепер містить **тільки корисну логіку**. Більше нічого зайвого.

---
## 9. Розширюваність: додавання нової ролі

Раніше додавання ролі `moderator` вимагало ручної правки в кількох функціях. Тепер додамо роль `superuser` — досить оновити список ролей у декораторі, тіло функції лишається незмінним.

In [6]:
# Просто оновлюємо декоратори — і все

@require_role("admin", "superuser")  # ← додали superuser тут
def delete_post_v3(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


@require_role("admin", "superuser")  # ← і тут
def archive_post_v3(post_id):
    print(f"📁  {current_user['name']} архівує пост #{post_id}")


# Тестуємо superuser
current_user = {"name": "Супер Юзер", "role": "superuser"}
delete_post_v3(99)
archive_post_v3(99)

🗑️  Супер Юзер видаляє пост #99
📁  Супер Юзер архівує пост #99


---
## 10. Висновок

### Що таке декоратор?

Декоратор — це функція, яка **загортає** іншу функцію і додає їй нову поведінку.

Ніякої магії немає. Це просто:
1. Функція, яка приймає функцію
2. Всередині робить щось додатково
3. Повертає нову функцію

```
🔧 require_role("admin")
        ↓
   загортає
        ↓
📦 delete_post
        ↓
  у нову функцію, яка
  спочатку перевіряє роль,
  потім (якщо ок) викликає оригінал
```

### Коли використовувати декоратори?

Якщо один і той самий шматок коду повторюється на початку або в кінці багатьох функцій — це сигнал розглянути декоратор.

Класичні приклади:
- Перевірка прав доступу (як у нас)
- Логування (записати в лог, що функція була викликана)
- Кешування (не рахувати те, що вже рахували)
- Вимірювання часу виконання
- Повторна спроба при помилці (retry)

### Головна ідея

> **Декоратор відповідає на питання «ЯК запустити функцію», а не «ЩО вона робить».**

Функція `delete_post` знає тільки одне: як видалити пост. Вона не повинна знати про ролі, логування, кешування. Це не її робота. Для цього є декоратори.

---
## 11. Додатковий приклад: `@timer`

Декоратори — загальна ідея, що виходить за межі перевірки доступу. Ось декоратор `timer`, який вимірює час виконання будь-якої функції:

In [7]:
# 🎓 Фінальний «живий" приклад для закріплення
# Напишемо декоратор-таймер — щоб зрозуміти, що декоратори — це загальна ідея

import time
import functools


def timer(func):
    """Вимірює час виконання функції"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()           # запам'ятовуємо час до
        result = func(*args, **kwargs) # виконуємо оригінальну функцію
        end = time.time()             # запам'ятовуємо час після
        print(f"⏱️  {func.__name__} виконалась за {end - start:.4f} сек")
        return result
    return wrapper


@timer
def heavy_calculation(n):
    """Симулюємо важкі обчислення"""
    total = sum(range(n))
    return total


@timer
def fast_calculation(n):
    """Швидка формула Гауса"""
    return n * (n - 1) // 2


print(heavy_calculation(1_000_000))
print(fast_calculation(1_000_000))

⏱️  heavy_calculation виконалась за 0.0170 сек
499999500000
⏱️  fast_calculation виконалась за 0.0000 сек
499999500000


---

## 12. Шпаргалка

```python
import functools

# Декоратор без параметрів
def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # робимо щось ДО
        result = func(*args, **kwargs)
        # робимо щось ПІСЛЯ
        return result
    return wrapper


# Декоратор З параметрами (фабрика)
def my_decorator_with_args(param):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # використовуємо param тут
            return func(*args, **kwargs)
        return wrapper
    return decorator


# Використання
@my_decorator
def foo():
    pass

@my_decorator_with_args("значення")
def bar():
    pass
```